# Vet record timeline

**What this notebook does:** reads the veterinary records of one pet, asks Gemini to list every
visit in them, and puts the visits from all the files into one timeline ordered by date.

**How it works, in order:**

1. The record files of one pet are found in BigQuery (query to be filled in) and downloaded to
   `data/records/<pet name or claim id>/`. PDF, JPG and PNG files are read.
2. Each file is sent to Gemini through Vertex AI (Google Cloud's service for running LLMs).
3. Gemini returns one entry per visit, in a fixed format (see "The output format" below).
4. The answer for each file is saved as a JSON file in `output/`.
5. The visits from all files are joined into one table, sorted by visit date.

**Before the first run:**

- Kernel: `.venv/bin/python` in this folder (created with `poetry install --no-root`).
- `gcloud auth application-default login` (signs this machine in to Google Cloud).
- Fill in the settings in the next cell.

In [2]:
from pathlib import Path

# --- Settings: fill these in ---

# The Google Cloud project that pays for the Gemini calls. Not confirmed: check which project to use.
PROJECT_ID = "FILL-IN"

# The Google Cloud region the calls run in. Must be in the EU (gdpr-compliance.md RULE 6).
# Not confirmed that every Gemini version is offered in this region.
LOCATION = "europe-west1"

# The Gemini versions to compare. The extraction runs once per file for each name in this list.
MODELS = ["gemini-3.7-flash"]

# The folder with one pet's record files, and the folder the answers are saved to.
PET = "FILL-IN"
RECORDS_DIR = Path("data/records") / PET
OUTPUT_DIR = Path("output") / PET

In [3]:
import json

import pandas as pd
from google import genai
from google.genai import types
from google.cloud import bigquery
from pydantic import BaseModel

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

## BigQuery


In [4]:
# BigQuery client (your own Google login) and a query helper that dry-runs first
bq_client = bigquery.Client(project='hedvig-coruscant')


def run_query(sql, max_gb=10):
    dry = bq_client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = dry.total_bytes_processed / 1e9
    print(f'Dry run: this query scans {gb:.2f} GB')
    if gb > max_gb:
        raise ValueError(f'{gb:.2f} GB is above max_gb={max_gb}. Approve by raising max_gb.')
    return bq_client.query(sql).result().to_dataframe()

/Users/ramshakhan/Coding/maker-days-document-processing/vet-record-reader/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [ ]:
# Emma's shortlist: one patient-history file for each of 30 randomly chosen first pet claims
SQL = """
WITH first_pet_claim AS (
SELECT contract_id, claim_id
FROM hedvig-coruscant.dbt_prod.claims_reporting
WHERE promise_category = 'Pet'
AND acquisition_source != 'manypets'
QUALIFY ROW_NUMBER() OVER (PARTITION BY contract_id ORDER BY notification_date, claim_id) = 1
)
SELECT f.id, f.claims_id, f.bucket, f.key, f.file_name, f.content_type, f.uploaded_at
FROM hedvig-coruscant.dbt_prod.stg_claims_service__claim_file f
JOIN first_pet_claim p ON p.claim_id = f.claims_id
WHERE f.file_name LIKE '%patient%'
AND DATE(f.uploaded_at) > '2025-01-01'
AND f.deleted_at IS NULL
AND f.marked_as_deleted_at IS NULL
AND f.upload_source IN ('MANUAL', 'APP')
QUALIFY ROW_NUMBER() OVER (PARTITION BY f.claims_id ORDER BY f.uploaded_at, f.id) = 1
ORDER BY FARM_FINGERPRINT(CAST(f.id AS STRING))
LIMIT 30
"""
record_files_table = run_query(SQL)

## File access on AWS

The uploaded claim files are stored on AWS S3. `dbt_prod.stg_claims_service__claim_file` has one row
per uploaded file, with `bucket` and `key` giving the file's location in S3. The query below returns
the patient-history file of one claim, as the test case for reading a file from S3.

In [5]:
CLAIM_ID = "e65784a6-5666-4e4b-9ff6-57c579d0abad"

SQL = f"""
SELECT claims_id, file_name, display_name, content_type, folder, bucket, key, uploaded_at
FROM `hedvig-coruscant.dbt_prod.stg_claims_service__claim_file`
WHERE claims_id = '{CLAIM_ID}'
  AND LOWER(file_name) LIKE '%patient_history%'
  AND NOT IFNULL(marked_as_deleted, FALSE)
"""
file_row = run_query(SQL)
file_row

Dry run: this query scans 0.12 GB


/Users/ramshakhan/Coding/maker-days-document-processing/vet-record-reader/.venv/lib/python3.12/site-packages/google/cloud/bigquery/client.py:440: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


,claims_id,file_name,display_name,content_type,folder,bucket,key,uploaded_at
0,e65784a6-5666-4e4b-9ff6-57c579d0abad,patient_history-2868715.pdf,patient_history-2868715.pdf,application/pdf,NaN,com-hedvig-claims-files-prod,claim-e65784a6-5666-4e4b-9ff6-57c579d0abad/1b7...,2024-11-04 10:16:43.419687+00:00


In [6]:
# Switch into the prod read-only role with your Ops credentials, then read files from S3.
# Read-only: get_object only reads a file; nothing in S3 is written, moved or deleted.
import json
import time
from pathlib import Path

import boto3

AWS_REGION = "eu-central-1"
SOURCE_PROFILE = "ops"      # your own user, in ~/.aws
ROLE_ARN = "arn:aws:iam::658549670687:role/CrossAccountReadOnlyRole"
CREDENTIALS_CACHE = Path("/tmp/.vet-record-reader-aws-credentials.json")
SESSION_HOURS = 1           # the role allows 1 hour today; set to 12 once #platform raises its maximum


def assume_read_only_role():
    """Ask AWS for temporary keys as CrossAccountReadOnlyRole, valid SESSION_HOURS.

    Asks for your MFA code. One request only: an MFA code can be used once, so no retries.
    """
    ops = boto3.Session(profile_name=SOURCE_PROFILE)
    mfa_serial = ops._session.full_config["profiles"].get("prod", {}).get("mfa_serial")
    kwargs = {"RoleArn": ROLE_ARN, "RoleSessionName": "vet-record-reader", "DurationSeconds": SESSION_HOURS * 3600}
    if mfa_serial:
        kwargs |= {"SerialNumber": mfa_serial, "TokenCode": input("AWS MFA code: ")}
    c = ops.client("sts").assume_role(**kwargs)["Credentials"]
    CREDENTIALS_CACHE.write_text(json.dumps({
        "AccessKeyId": c["AccessKeyId"], "SecretAccessKey": c["SecretAccessKey"],
        "SessionToken": c["SessionToken"], "Expiration": c["Expiration"].timestamp(),
    }))
    print(f"Credentials valid until {c['Expiration']:%Y-%m-%d %H:%M %Z}")


def prod_session():
    """A boto3 session as the read-only role, reusing cached keys until they expire."""
    if not CREDENTIALS_CACHE.exists() or json.loads(CREDENTIALS_CACHE.read_text())["Expiration"] < time.time() + 60:
        assume_read_only_role()
    c = json.loads(CREDENTIALS_CACHE.read_text())
    return boto3.Session(
        aws_access_key_id=c["AccessKeyId"],
        aws_secret_access_key=c["SecretAccessKey"],
        aws_session_token=c["SessionToken"],
        region_name=AWS_REGION,
    )


def fetch_file(bucket, key):
    """Return the file's bytes, to send straight to Gemini."""
    return prod_session().client("s3").get_object(Bucket=bucket, Key=key)["Body"].read()


def download_file(bucket, key, to_dir=Path("data/records")):
    """Save the file locally (data/ is ignored by git) and return its path, to open and look at."""
    to_dir.mkdir(parents=True, exist_ok=True)
    path = to_dir / Path(key).name
    path.write_bytes(fetch_file(bucket, key))
    return path


# Test on the row from the query above
row = file_row.iloc[0]
file_bytes = fetch_file(row["bucket"], row["key"])
print(f"{row['file_name']}: {len(file_bytes) / 1e6:.2f} MB, content_type={row['content_type']}")

Credentials valid until 2026-11-18 13:49 UTC
patient_history-2868715.pdf: 0.48 MB, content_type=application/pdf


In [7]:
path = download_file(row["bucket"], row["key"])
print(path.resolve())


/Users/ramshakhan/Coding/maker-days-document-processing/vet-record-reader/data/records/1b740b64-f3f5-420a-a104-5bbc93e44134-patient_history-2868715.pdf


path = download_file(row["bucket"], row["key"])
print(path.resolve())


## Gemini on a file

Sends a file and a prompt to Gemini through Vertex AI in Hedvig's Google Cloud project, EU region.
The file's bytes travel inside the request; nothing is uploaded to a bucket or kept as a file.

In [23]:
# --- Config: every input to the Gemini call ---
GEMINI_PROJECT = "hedvig-coruscant"   # the Google Cloud project claims-service uses for Gemini
GEMINI_LOCATION = "eu"
GEMINI_MODEL = "gemini-3.8-flash"
TEMPERATURE = 0                      # 0 = the same answer for the same input, as far as Gemini allows
PROMPT = "List every visit in this veterinary record, oldest first. Keep the record's own language."
LABELS = {"project": "vet-record-reader", "team": "maker-days-2026"}   # shown against the cost in Cloud Billing

# Prices for gemini-3.8-flash on the Vertex AI EU endpoint, USD per million tokens. Third-party listings
# (requesty.ai, sim.ai), read 18 Nov 2026; promotional rates until 31 Dec 2026, listed to double from 1 Jan 2027.
PRICE_PER_M_INPUT_USD = 0.83
PRICE_PER_M_OUTPUT_USD = 4.13        # thinking tokens are billed as output

# The shape Gemini must answer in: one entry per visit
from pydantic import BaseModel


class Visit(BaseModel):
    date: str            # YYYY-MM-DD
    clinic: str
    reason: str
    diagnoses: list[str]


class Visits(BaseModel):
    visits: list[Visit]


OUTPUT_FORMAT = Visits               # or None for free text


In [24]:
from google import genai
from google.genai import types

gemini = genai.Client(vertexai=True, project=GEMINI_PROJECT, location=GEMINI_LOCATION)


def ask_gemini(file_bytes, content_type, prompt, model=GEMINI_MODEL, output_format=OUTPUT_FORMAT):
    """Send one file and a prompt to Gemini. Returns (answer, usage).

    answer: text, or an instance of output_format when one is given.
    usage: token counts for the call: input, output, thinking, total.
    """
    config = types.GenerateContentConfig(temperature=TEMPERATURE, labels=LABELS)
    if output_format is not None:
        config.response_mime_type = "application/json"
        config.response_schema = output_format
    response = gemini.models.generate_content(
        model=model,
        contents=[types.Part.from_bytes(data=file_bytes, mime_type=content_type), prompt],
        config=config,
    )
    u = response.usage_metadata
    usage = {
        "input_tokens": u.prompt_token_count,
        "output_tokens": u.candidates_token_count,
        "thinking_tokens": u.thoughts_token_count,
        "total_tokens": u.total_token_count,
    }
    usage["cost_usd"] = round(
        (usage["input_tokens"] * PRICE_PER_M_INPUT_USD
         + (usage["output_tokens"] + (usage["thinking_tokens"] or 0)) * PRICE_PER_M_OUTPUT_USD) / 1e6, 4)
    answer = response.parsed if output_format is not None else response.text
    return answer, usage

In [25]:
# Test on the file read from S3 above: one row per visit, oldest first, saved as a CSV
answer, usage = ask_gemini(file_bytes, row["content_type"], PROMPT)
visits = pd.DataFrame([v.model_dump() for v in answer.visits])
visits["diagnoses"] = visits["diagnoses"].str.join("; ")
visits = visits.sort_values("date").reset_index(drop=True)
out = Path("output") / f"{row['claims_id']}_visits.csv"
out.parent.mkdir(exist_ok=True)
visits.to_csv(out, index=False)
print(" | ".join(f"{k}: {v}" for k, v in usage.items()), "| saved to", out)
visits

/Users/ramshakhan/Coding/maker-days-document-processing/vet-record-reader/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


input_tokens: 28922 | output_tokens: 2229 | thinking_tokens: 3915 | total_tokens: 35066 | cost_usd: 0.0494 | saved to output/e65784a6-5666-4e4b-9ff6-57c579d0abad_visits.csv


,date,clinic,reason,diagnoses
0,2021-06-01,Täby Djursjukhus Evidensia,"Vaccination 12-veckor valp/kattunge, ett djur",Vaccination
1,2021-06-21,Evidensias Onlineveterinär,MV: ögonbesvär,"Konjunktivit, mukopurulent/purulent, akut"
2,2021-08-02 - 2021-08-09,Evidensias Onlineveterinär,MV: klåda på kroppen,Klåda
3,2021-08-09,Täby Djursjukhus Evidensia,MV: klåda 1v och fortsatt röd i ögonen,Mjäll
4,2021-08-26,Täby Djursjukhus Evidensia,åb hud,Klåda med hudlesioner
5,2021-12-29,Täby Djursjukhus Evidensia,påbörja allergiutredning.,Klåda
6,2022-03-02,Täby Djursjukhus Evidensia,"Vaccination, vill bara vaccinera det övriga va...",Vaccination
7,2022-08-25,Täby Djursjukhus Evidensia,"Veterinärkonsultation hund/katt, ett djur - hu...",Pyodermi; Klåda; Malasseziainfektion; Pododerm...
8,2022-09-13,Täby Djursjukhus Evidensia,"Åb öron, tassar,-allergi",Otitis externa; Pododermatit; Klåda
9,2022-09-26,Täby Djursjukhus Evidensia,ÅB öron tassar hud,Klåda
